# 03 — Pull training imagery from Earth Engine

**Purpose.** Downloads the Landsat imagery used to *train* the model. For each of the ~18k glaciers
from notebook 01, it pulls every Landsat 5, 7 and 8 scene within ±50 days of the date the glacier's
GLIMS outline was drawn, so each scene can be paired with that outline as its label (notebooks
09–10). `EePull` (`glacierview.earthengine`) applies `CLOUD_COVER <= 10` and image quality 9 over
each bounding box, writes one 30 m GeoTIFF per scene, and logs a line per scene. A satellite that
fails for a glacier is logged as a warning and skipped.

The pull resumes: glaciers that already have a scene folder are skipped. Glaciers are shuffled, so a
partial pull still covers every region.

## Inputs
- **`config.glims_selection_path()`**: bounding boxes and outline dates from notebook 01.
- **Earth Engine account**: `ee.Authenticate()` opens a browser sign-in; `ee.Initialize` needs a
  Cloud project with Earth Engine enabled.
- **Environment**: the project's uv environment as the kernel. A long, network-bound run.

## Outputs
Under `config.landing_zone(config.TRAINING_DATA_LABEL)` (gitignored):
- `landsat/<glims_id>/<glims_id>_<date>_<L5|L7|L8>_C02_T1_L2_SR.tif`: one file per scene.
- `landsat/<glims_id>/meta_data/metadata_list_l{5,7,8}`: raw Earth Engine metadata, read by notebook 06.
- `logs/log_full_training_data.log`: one line per scene, read by notebook 06.

Training DEMs come from notebook 04 with `LABEL = config.TRAINING_DATA_LABEL`.

## Manual step afterwards (not in code)
The canonical copy lives in `s3://raw-training-images-t1-l2-sr` (us-west-1) as `landsat/<glims_id>/`
and `logs/`. Upload new pulls there. See `.agents/references/aws.md`.

In [ ]:
from datetime import datetime, timedelta

import ee
from tqdm import tqdm

from glacierview import config, glims, log, metadata
from glacierview.earthengine import EePull

log.configure()
ee.Authenticate()
ee.Initialize(project="psyched-upgrade-377000")

In [ ]:
LABEL = config.TRAINING_DATA_LABEL
WINDOW = timedelta(days=50)

pull = EePull(config.log_dir(LABEL), "full_training_data")
landsat = config.landsat_dir(LABEL)
done = set(metadata.glacier_ids(landsat)) if landsat.exists() else set()

glaciers = glims.read_selection().sample(frac=1, random_state=0)
glaciers = glaciers[~glaciers["glac_id"].isin(done)]
print(f"{len(glaciers)} glaciers to pull, {len(done)} already done")

In [ ]:
for glacier in tqdm(glaciers.itertuples(), total=len(glaciers)):
    outline_date = datetime.fromisoformat(glacier.src_date[:10])
    pull.export_all_landsat(
        glacier.glac_id,
        glims.parse_bounding_box(glacier.bboxes),
        outline_date - WINDOW,
        outline_date + WINDOW,
        config.glacier_dir(glacier.glac_id, LABEL),
    )